# 코드 프로파일링 및 타이밍

코드 개발 및 데이터 처리 파이프라인 생성 과정에서 다양한 구현 간에 절충안을 만들어야 하는 경우가 많습니다.
알고리즘 개발 초기에는 이러한 사항에 대해 걱정하는 것이 비생산적일 수 있습니다. 도널드 커누스가 유명하게 말했듯이, "우리는 작은 효율성은 잊어버려야 합니다. 약 97%의 경우에 말입니다. 섣부른 최적화는 모든 악의 근원입니다."

그러나 코드가 작동하면 효율성을 좀 더 자세히 살펴보는 것이 유용할 수 있습니다.
주어진 명령 또는 명령 집합의 실행 시간을 확인하는 것이 유용할 때도 있고, 여러 줄로 된 프로세스를 검토하여 복잡한 일련의 작업에서 병목 현상이 발생하는 위치를 파악하는 것이 유용할 때도 있습니다.
IPython은 이러한 종류의 코드 타이밍 및 프로파일링을 위한 광범위한 기능에 대한 액세스를 제공합니다.
여기서는 다음 IPython 매직 명령에 대해 설명합니다.

- `%time`: 단일 문의 실행 시간 측정
- `%timeit`: 정확도를 높이기 위해 단일 문의 반복 실행 시간 측정
- `%prun`: 프로파일러로 코드 실행
- `%lprun`: 줄 단위 프로파일러로 코드 실행
- `%memit`: 단일 문의 메모리 사용량 측정
- `%mprun`: 줄 단위 메모리 프로파일러로 코드 실행

마지막 네 가지 명령은 IPython에 번들로 제공되지 않습니다. 사용하려면 `line_profiler` 및 `memory_profiler` 확장을 가져와야 하며, 다음 섹션에서 이에 대해 설명합니다.

## 코드 스니펫 타이밍: %timeit 및 %time

[IPython 매직 명령어](01.03-Magic-Commands.ipynb)의 매직 함수 소개에서 `%timeit` 라인 매직과 `%%timeit` 셀 매직을 보았습니다. 이러한 매직은 코드 스니펫의 반복 실행 시간을 측정하는 데 사용할 수 있습니다.

In [1]:
%timeit sum(range(100))

루프당 1.53 µs ± 47.8 ns (7회 실행, 각 1,000,000 루프의 평균 ± 표준 편차)


이 작업은 매우 빠르기 때문에 `%timeit`은 자동으로 많은 반복을 수행합니다.
느린 명령의 경우 `%timeit`은 자동으로 조정되어 반복 횟수를 줄입니다.

In [2]:
%%timeit
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

루프당 536ms ± 15.9ms (7회 실행, 각 1루프의 평균 ± 표준 편차)


때로는 작업을 반복하는 것이 최선의 선택이 아닐 수도 있습니다.
예를 들어 정렬하려는 목록이 있는 경우 반복 작업으로 인해 오해를 받을 수 있습니다. 미리 정렬된 목록을 정렬하는 것이 정렬되지 않은 목록을 정렬하는 것보다 훨씬 빠르므로 반복하면 결과가 왜곡됩니다.

In [3]:
import random
L = [random.random() for i in range(100000)]
%timeit L.sort()

루프당 1.71ms ± 334µs (7회 실행, 각 1000루프의 평균 ± 표준 편차)


이 경우 `%time` 매직 함수가 더 나은 선택일 수 있습니다. 또한 짧은 시스템 관련 지연이 결과에 영향을 미칠 가능성이 없는 장기 실행 명령에도 좋은 선택입니다.
정렬되지 않은 목록과 미리 정렬된 목록의 정렬 시간을 측정해 보겠습니다.

In [4]:
import random
L = [random.random() for i in range(100000)]
print("정렬되지 않은 목록 정렬:")
%time L.sort()

정렬되지 않은 목록 정렬:
CPU 시간: 사용자 31.3ms, 시스템: 686µs, 총: 32ms
벽 시간: 33.3ms


In [5]:
print("이미 정렬된 목록 정렬:")
%time L.sort()

이미 정렬된 목록 정렬:
CPU 시간: 사용자 5.19ms, 시스템: 268µs, 총: 5.46ms
벽 시간: 14.1ms


미리 정렬된 목록이 정렬되는 속도가 얼마나 빠른지 확인하십시오. 그러나 미리 정렬된 목록의 경우에도 `%time`과 `%timeit`의 타이밍이 얼마나 오래 걸리는지 확인하십시오!
이는 `%timeit`이 시스템 호출이 타이밍을 방해하는 것을 방지하기 위해 내부적으로 몇 가지 영리한 작업을 수행하기 때문입니다.
예를 들어, 타이밍에 영향을 미칠 수 있는 사용되지 않는 파이썬 객체 정리(가비지 수집이라고 함)를 방지합니다.
이러한 이유로 `%timeit` 결과는 일반적으로 `%time` 결과보다 눈에 띄게 빠릅니다.

`%time`의 경우 `%timeit`과 마찬가지로 `%%` 셀 매직 구문을 사용하면 여러 줄 스크립트의 타이밍을 측정할 수 있습니다.

In [6]:
%%time
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

CPU 시간: 사용자 655ms, 시스템: 5.68ms, 총: 661ms
벽 시간: 710ms


`%time` 및 `%timeit`과 사용 가능한 옵션에 대한 자세한 내용은 IPython 도움말 기능을 사용하십시오(예: IPython 프롬프트에 `%time?` 입력).

## 전체 스크립트 프로파일링: %prun

프로그램은 많은 단일 문으로 구성되며 때로는 이러한 문을 컨텍스트에서 타이밍하는 것이 자체적으로 타이밍하는 것보다 더 중요합니다.
파이썬에는 기본 제공 코드 프로파일러가 포함되어 있지만(파이썬 설명서에서 읽을 수 있음) IPython은 매직 함수 `%prun`의 형태로 이 프로파일러를 사용하는 훨씬 편리한 방법을 제공합니다.

예를 들어 몇 가지 계산을 수행하는 간단한 함수를 정의해 보겠습니다.

In [7]:
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total

이제 함수 호출과 함께 `%prun`을 호출하여 프로파일링된 결과를 볼 수 있습니다.

In [8]:
%prun sum_of_lists(1000000)

         0.932초에 14개의 함수 호출

   내부 시간순으로 정렬됨

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        5    0.808    0.162    0.808    0.162 <ipython-input-7-f105717832a2>:4(<listcomp>)
        5    0.066    0.013    0.066    0.013 {built-in method builtins.sum}
        1    0.044    0.044    0.918    0.918 <ipython-input-7-f105717832a2>:1(sum_of_lists)
        1    0.014    0.014    0.932    0.932 <string>:1(<module>)
        1    0.000    0.000    0.932    0.932 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}

결과는 각 함수 호출에 대한 총 시간 순서대로 실행이 가장 많은 시간을 소비하는 위치를 나타내는 테이블입니다. 이 경우 실행 시간의 대부분은 `sum_of_lists` 내의 목록 이해에 있습니다.
여기서부터 알고리즘의 성능을 향상시키기 위해 어떤 변경을 할 수 있는지 생각하기 시작할 수 있습니다.

`%prun` 및 사용 가능한 옵션에 대한 자세한 내용은 IPython 도움말 기능을 사용하십시오(즉, IPython 프롬프트에 `%prun?` 입력).

## %lprun을 사용한 줄별 프로파일링

`%prun`의 함수별 프로파일링은 유용하지만 때로는 줄별 프로파일 보고서가 더 편리할 때가 있습니다.
이것은 파이썬이나 IPython에 내장되어 있지 않지만 이 작업을 수행할 수 있는 `line_profiler` 패키지를 설치할 수 있습니다.
파이썬의 패키징 도구인 `pip`를 사용하여 `line_profiler` 패키지를 설치하는 것으로 시작합니다:

```
$ pip install line_profiler
```

다음으로 IPython을 사용하여 이 패키지의 일부로 제공되는 `line_profiler` IPython 확장을 로드할 수 있습니다.

In [9]:
%load_ext line_profiler

이제 `%lprun` 명령은 모든 함수의 줄별 프로파일링을 수행합니다. 이 경우 프로파일링하려는 함수를 명시적으로 지정해야 합니다.

In [10]:
%lprun -f sum_of_lists sum_of_lists(5000)

타이머 단위: 1e-06초

총 시간: 0.014803초
파일: <ipython-input-7-f105717832a2>
함수: 1행의 sum_of_lists

줄 #      적중 횟수         시간  적중당   % 시간  줄 내용
     1                                           def sum_of_lists(N):
     2         1          6.0      6.0      0.0      total = 0
     3         6         13.0      2.2      0.1      for i in range(5):
     4         5      14242.0   2848.4     96.2          L = [j ^ (j >> i) for j in range(N)]
     5         5        541.0    108.2      3.7          total += sum(L)
     6         1          1.0      1.0      0.0      return total

상단의 정보는 결과를 읽는 열쇠를 제공합니다. 시간은 마이크로초 단위로 보고되며 프로그램이 가장 많은 시간을 소비하는 위치를 확인할 수 있습니다.
이 시점에서 이 정보를 사용하여 스크립트의 측면을 수정하여 원하는 사용 사례에 대해 더 나은 성능을 발휘하도록 할 수 있습니다.

`%lprun` 및 사용 가능한 옵션에 대한 자세한 내용은 IPython 도움말 기능을 사용하십시오(즉, IPython 프롬프트에 `%lprun?` 입력).

## 메모리 사용량 프로파일링: %memit 및 %mprun

프로파일링의 또 다른 측면은 작업에서 사용하는 메모리 양입니다.
이것은 다른 IPython 확장인 `memory_profiler`로 평가할 수 있습니다.
`line_profiler`와 마찬가지로 `pip`를 사용하여 확장을 설치하는 것으로 시작합니다:

```
$ pip install memory_profiler
```

그런 다음 IPython을 사용하여 로드할 수 있습니다.

In [11]:
%load_ext memory_profiler

메모리 프로파일러 확장에는 두 가지 유용한 매직 함수가 포함되어 있습니다. `%memit`(`%timeit`에 해당하는 메모리 측정 기능 제공) 및 `%mprun`(`%lprun`에 해당하는 메모리 측정 기능 제공).
`%memit` 매직 함수는 매우 간단하게 사용할 수 있습니다.

In [12]:
%memit sum_of_lists(1000000)

최대 메모리: 141.70MiB, 증가량: 75.65MiB


이 함수는 약 140MB의 메모리를 사용하는 것을 볼 수 있습니다.

메모리 사용량에 대한 줄별 설명을 보려면 `%mprun` 매직 함수를 사용할 수 있습니다.
안타깝게도 이것은 노트북 자체가 아닌 별도의 모듈에 정의된 함수에 대해서만 작동하므로 `%%file` 셀 매직을 사용하여 `mprun_demo.py`라는 간단한 모듈을 만드는 것으로 시작하겠습니다. 이 모듈에는 `sum_of_lists` 함수가 포함되어 있으며 메모리 프로파일링 결과를 더 명확하게 해주는 한 가지 추가 사항이 있습니다.

In [13]:
%%file mprun_demo.py
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
        del L # L에 대한 참조 제거
    return total

mprun_demo.py 덮어쓰기


이제 이 함수의 새 버전을 가져와서 메모리 줄 프로파일러를 실행할 수 있습니다.

In [14]:
from mprun_demo import sum_of_lists
%mprun -f sum_of_lists sum_of_lists(1000000)

파일 이름: /Users/jakevdp/github/jakevdp/PythonDataScienceHandbook/notebooks_v2/mprun_demo.py

줄 번호    메모리 사용량    증가량  발생 횟수   줄 내용
     1     66.7 MiB     66.7 MiB           1   def sum_of_lists(N):
     2     66.7 MiB      0.0 MiB           1       total = 0
     3     75.1 MiB      8.4 MiB           6       for i in range(5):
     4    105.9 MiB     30.8 MiB     5000015           L = [j ^ (j >> i) for j in range(N)]
     5    109.8 MiB      3.8 MiB           5           total += sum(L)
     6     75.1 MiB    -34.6 MiB           5           del L # L에 대한 참조 제거
     7     66.9 MiB     -8.2 MiB           1       return total

여기서 `Increment` 열은 각 줄이 총 메모리 예산에 얼마나 영향을 미치는지 알려줍니다. 목록 `L`을 만들고 삭제할 때 약 30MB의 메모리 사용량이 추가되는 것을 관찰하십시오.
이것은 파이썬 인터프리터 자체의 백그라운드 메모리 사용량 위에 있습니다.

`%memit` 및 `%mprun`과 사용 가능한 옵션에 대한 자세한 내용은 IPython 도움말 기능을 사용하십시오(예: IPython 프롬프트에 `%memit?` 입력).